# OC14 — Merge the SFT (Base) adapter → 16-bit weights

Loads the SFT LoRA adapter from the SFT kernel output and writes ordinary 16-bit weights that vLLM
can serve with no special flags. (The SFT run saved only the adapter; merge happens here.)

In [ ]:
# Official Unsloth Kaggle install (verbatim, June 2026): install torch from the cu128
# index FIRST so its wheels carry the T4/sm_75 CUDA kernels. Plain `pip install unsloth`
# let pip resolve a torch build lacking them -> 'CUDA: no kernel image' at model load.
!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import subprocess, sys
with open('/kaggle/working/requirements-train.lock.txt', 'w') as fh:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=fh)
print('lockfile written ->', '/kaggle/working/requirements-train.lock.txt')

In [ ]:
import glob, os
ad = glob.glob('/kaggle/input/**/sft_adapter/adapter_config.json', recursive=True)
assert ad, 'SFT adapter not found — attach the SFT kernel as a kernel source'
SFT_ADAPTER_DIR = os.path.dirname(ad[0]); print('SFT_ADAPTER_DIR =', SFT_ADAPTER_DIR)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_DIR, max_seq_length=2048, load_in_4bit=True)
OUT = '/kaggle/working/sft_merged_16bit'
model.save_pretrained_merged(OUT, tokenizer, save_method='merged_16bit')
files = os.listdir(OUT); print('merged files:', files)
assert any(f.endswith('.safetensors') for f in files), 'merge wrote no weights!'
print('OK: SFT merged 16-bit weights at', OUT)